# From TERRA embeddings to H&E-only cell classification with CellBridge

Use **TERRA as a molecular teacher**: extract cell representations from paired spatial transcriptomics, then teach an image encoder to learn from them. With **CellBridge**, the resulting encoder can be used to classify cells from H&E images alone.

This tutorial extends the [TERRA zero-shot quickstart](https://terra-st.readthedocs.io/en/latest/notebooks/zero_shot_quickstart.html) to cross-modal learning. We use **HEST H&E–Xenium pairs** to explain training, then **released CellBridge weights and NuCLS** for inference.

| Stage | Inputs | Output |
|---|---|---|
| Frozen TERRA teacher | Raw counts + native spatial coordinates | 384-dimensional cell embeddings |
| Image encoder adaptation | Registered H&E crops + matching TERRA embeddings | UNI2-h LoRA adapters + an alignment head |
| Task classifier fitting | Adapted image features + reference labels | A classifier for your cell-type vocabulary |
| H&E-only prediction | H&E + detected cell locations | Labels and classifier probabilities |
| Optional Cell-Ontology-Fallback (COF) | Reference labels, calibration cells, and an ontology | Calibrated prediction sets |

**Choose your route:**

- **Pretrained quickstart:** leave `PREPARE_PAIRED = False`; run setup, then continue at section 3.
- **Learn from paired data:** enable `PREPARE_PAIRED` and run section 2 to prepare TERRA targets. Encoder training is explained and a terminal command is provided, but the notebook never launches it.
- **Prediction sets:** enable `RUN_COF` to add the optional section 7.

The notebook fits a small task classifier. New H&E slides also need cell locations from a detector or segmentation pipeline; the released encoder does not supply a universal label vocabulary or predict gene counts.

## 1. Setup and model access

Use a **Python 3.11** environment and install a [PyTorch build for your GPU](https://pytorch.org/get-started/locally/) first. From the repository root:

```bash
python -m pip install -e '.[models,preprocess,analysis]'
python -m pip install -r requirements/encoders.txt
python -m ipykernel install --user --name cellbridge --display-name 'CellBridge (Python 3.11)'
python -m jupyterlab notebooks/TUTORIAL.ipynb
```

For the optional TERRA section, also install `python -m pip install terra-st` following the [TERRA installation guide](https://terra-st.readthedocs.io/en/latest/installation.html). TERRA requires an NVIDIA GPU. If TERRA uses a separate environment, run setup and sections 2.1–2.3 there and reuse the saved NPZ files in the CellBridge environment.

Request access to [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h), and, for section 2, accept the [HEST dataset terms](https://huggingface.co/datasets/MahmoodLab/hest). Authenticate with a read token below. The [CellBridge adapter](https://huggingface.co/ogutsevda/cellbridge) is public; UNI2-h still requires its own access approval.

A GPU is recommended for image encoding. NuCLS downloads about 0.8 GB; UNI2-h weights and extracted data need additional space. The optional HEST whole slides and transcript tables can be much larger. All tutorial outputs stay under `outputs/tutorial/`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from PIL import Image
from huggingface_hub import get_token, hf_hub_download, notebook_login

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/train/lora/cellbridge.yaml").is_file())
WORK = ROOT / "outputs" / "tutorial"
PAIRED = WORK / "data/train/hest"
NUCLS = WORK / "data/eval/nucls"
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREPARE_PAIRED = False  # Optional TERRA teacher preparation (section 2).
RUN_COF = False  # Optional ontology-aware prediction sets (section 7).

os.environ.update({
    "CELLBRIDGE_ROOT": str(ROOT),
    "CELLBRIDGE_DATA_ROOT": str(WORK / "data"),
    "CELLBRIDGE_OUTPUTS_ROOT": str(WORK / "results"),
    "CELLBRIDGE_CHECKPOINTS_ROOT": str(WORK / "checkpoints"),
})
print(f"Device: {DEVICE} | Outputs: {WORK}")

In [ ]:
if get_token() is None:
    notebook_login()

## 2. Learn from paired data with TERRA (optional)

TERRA converts a cell's expression profile and spatial context into molecular representations. Here those representations become **fixed supervision for a second modality**: H&E. TERRA remains frozen; training updates the image encoder's adapters and alignment head. No cell-type annotations are needed for this stage.

```text
raw counts + spatial coordinates → frozen TERRA → cell embedding ─┐
                                                                ├→ contrastive alignment
registered H&E crop → UNI2-h + trainable LoRA → projection ────────┘
```

### 2.1. Prepare spatial counts paired with H&E

We use two human Xenium sections from HEST: `TENX96` for encoder training and `TENX99` for validation, matching the repository's section split. This small cohort demonstrates the workflow, not the released model's performance. For your experiment, keep all sections from a donor/patient in one split and reserve an independent test cohort.

Download the metadata, registered H&E, and aligned transcript tables needed for these sections. HEST supplies the correspondence between modalities; its spot patches are not the cell-centered pairs used here. See the [HEST dataset card](https://huggingface.co/datasets/MahmoodLab/hest) for the file layout.

In [ ]:
if PREPARE_PAIRED:
    from huggingface_hub import snapshot_download

    HEST_RAW = WORK / "hest_raw"
    sample_ids = ["TENX96", "TENX99"]
    snapshot_download(
        "MahmoodLab/hest", repo_type="dataset", local_dir=str(HEST_RAW),
        allow_patterns=["HEST_v1_3_0.csv"] + [
            name for sid in sample_ids for name in (
                f"wsis/{sid}.tif", f"transcripts/{sid}_transcripts.parquet",
            )
        ],
    )
    hest_meta = pd.read_csv(HEST_RAW / "HEST_v1_3_0.csv", low_memory=False).set_index("id")
    PAIRED.mkdir(parents=True, exist_ok=True)
    manifest = pd.DataFrame([{
        "id": sid,
        "wsi": str(HEST_RAW / f"wsis/{sid}.tif"),
        "transcripts": str(HEST_RAW / f"transcripts/{sid}_transcripts.parquet"),
        "pixel_size_um": float(hest_meta.loc[sid, "pixel_size_um_estimated"]),
        "output_dir": str(PAIRED / sid),
    } for sid in sample_ids])
    manifest.to_csv(PAIRED / "manifest.csv", index=False)
    pd.DataFrame({
        "sample_id": sample_ids, "split": ["distill_train", "distill_val"],
    }).to_csv(PAIRED / "splits.csv", index=False)
    PREPROCESS_CONFIG = WORK / "preprocess.yaml"
    PREPROCESS_CONFIG.write_text(yaml.safe_dump({
        "output_root": str(PAIRED), "qv_threshold": 20,
    }))
    display(manifest[["id", "pixel_size_um"]])

Reconstruct raw counts with CellBridge's streaming QC command. It removes low-quality calls, control transcripts, and unassigned transcripts, and derives each cell's location from its retained transcripts.

For **your own paired data**, start with an AnnData file with this schema and a manifest like the one above; skip the HEST download and QC command. Registration must already map the same cells into H&E coordinates.

| Field | Meaning |
|---|---|
| `X`, `var_names` | Raw gene counts and gene names; do not log-normalize |
| `obs_names`, `obs['cell_id']` | Matching unique string cell IDs |
| `obsm['spatial']` | Native spatial coordinates in microns for TERRA |
| `obs['he_x']`, `obs['he_y']` | Registered cell centers in level-0 H&E pixels |
| Manifest `pixel_size_um` | Microns per level-0 H&E pixel |

Keep native spatial coordinates separate from image pixels. TERRA should process each section independently so neighborhoods cannot cross sections.

In [ ]:
if PREPARE_PAIRED:
    for sid in sample_ids:
        subprocess.run([
            sys.executable, "-m", "cellbridge.preprocess.02_quality_check_streaming",
            "--config", str(PREPROCESS_CONFIG), "--sample-id", sid,
        ], check=True)

### 2.2. Download the pretrained TERRA teacher

`download_pretrained` returns a local bundle containing the model weights, tokenizer configuration, and gene-reference files. Use the same bundle for all training and validation sections so their teacher vectors share a representation space.

This follows the [TERRA quickstart](https://terra-st.readthedocs.io/en/latest/notebooks/zero_shot_quickstart.html). We expose its individual preprocessing steps below to make cell identity and H&E pairing explicit.

In [ ]:
if PREPARE_PAIRED:
    import anndata as ad
    from terra import download_pretrained, harmonize_adata, tokenize_adata, embed_dataset

    assert torch.cuda.is_available(), "Run TERRA in an NVIDIA GPU environment."
    terra_dir = Path(download_pretrained("Lotfollahi-lab/TERRA-112M"))

### 2.3. Harmonize → tokenize → embed → export paired targets

Harmonization matches genes to TERRA's vocabulary and filters low-support genes and cells. Tokenization builds the spatial input sequences, and embedding runs the pretrained teacher. Process each section independently; do not mix coordinate systems or build neighborhoods across sections.

Harmonization can remove cells, so recover H&E coordinates **by the tokenized cell IDs**, never by the original row positions. Embed the section before subsampling training pairs to preserve spatial context.

Already have a TERRA-embedded AnnData from the quickstart? You can reuse `adata.obsm['cell_emb']` instead of rerunning TERRA, provided its cells have registered `he_x, he_y` coordinates. Export `cell_ids=adata.obs_names`, `cell_emb=adata.obsm['cell_emb']`, and those coordinates in the NPZ schema below, keeping their row order identical. Use targets from the same TERRA bundle and preprocessing across the cohort.

In [ ]:
if PREPARE_PAIRED:
    for sid in sample_ids:
        sample_dir = PAIRED / sid
        adata = ad.read_h5ad(sample_dir / f"{sid}.h5ad")
        assert adata.obs_names.is_unique
        adata.obs["cell_id"] = adata.obs_names.astype(str)
        # Harmonize raw counts against this teacher's gene vocabulary.
        harmonized = harmonize_adata(
            adata,
            gene_mapping_dict_file_path=str(terra_dir / "ensembl_dictionary.pkl"),
            gene_occurrence_count_file_path=str(terra_dir / "gene_count_dictionary.pkl"),
            min_cells_per_gene=10, min_genes_per_cell=10,
        )
        # Build spatial neighborhoods within this section only.
        tokens = tokenize_adata(
            adata=harmonized, model_folder_path=str(terra_dir),
            cache_directory_path=str(sample_dir / "tokenize_cache"), nproc=4,
        )
        # Run frozen TERRA, then export the cell-level alignment targets.
        teacher = embed_dataset(
            dataset=tokens, model_folder_path=str(terra_dir), batch_size=128, num_workers=4,
        )
        cell_ids = np.asarray(tokens["cell_id"], dtype=str)
        centers = harmonized.obs.loc[cell_ids, ["he_x", "he_y"]]
        features = np.asarray(teacher["cell_emb"], dtype=np.float32)
        assert len(np.unique(cell_ids)) == len(cell_ids)
        assert features.shape == (len(cell_ids), 384)
        assert np.isfinite(features).all() and np.isfinite(centers.to_numpy()).all()
        np.savez(
            sample_dir / f"{sid}_terra.npz", cell_ids=cell_ids, cell_emb=features,
            he_x=centers.he_x.to_numpy(), he_y=centers.he_y.to_numpy(),
        )
        print(f"{sid}: saved {features.shape} teacher features")

TERRA returns both `cell_emb` and `neighborhood_emb`; CellBridge uses **`cell_emb`** as its teacher target. These vectors are learned representations, not cell-type labels or reconstructed gene counts. Only the cell targets and their registered centers need to cross into the image-training environment.

The resulting training inputs are small compared with the transcript tables:

```text
hest/
  manifest.csv                 # id, wsi, pixel_size_um, output_dir
  splits.csv                   # sample_id, split (distill_train / distill_val)
  TENX96/TENX96_terra.npz       # cell_ids, cell_emb, he_x, he_y
  TENX99/TENX99_terra.npz
```

`HESTCellDataset` reads these targets and crops the WSI lazily. Each training crop covers **112 × 112 µm**, resized to **224 × 224 pixels** (0.5 µm/pixel). The image and TERRA vector must describe the same cell. Cell-type labels are not required for this contrastive stage.

In [ ]:
if PREPARE_PAIRED:
    from cellbridge.train.lora.data import crop_cell, open_wsi

    sid = sample_ids[0]
    row = manifest.set_index("id").loc[sid]
    with np.load(PAIRED / sid / f"{sid}_terra.npz", allow_pickle=False) as saved:
        cx, cy = float(saved["he_x"][0]), float(saved["he_y"][0])
    wsi = open_wsi(row.wsi)
    try:
        display(crop_cell(wsi, cx, cy, row.pixel_size_um))
    finally:
        wsi.store.close()

### 2.4. Transfer TERRA representations to the image encoder

CellBridge starts from pretrained UNI2-h, freezes its original weights, and trains rank-8 LoRA updates in its last four attention blocks plus a projection head. The head maps 1536-dimensional image features into the teacher's 384-dimensional space. Symmetric InfoNCE pairs each crop with **the same cell's TERRA vector** and contrasts it with other cells in the batch.

TERRA supplies molecular supervision throughout training, but is no longer needed for H&E-only prediction. Cached TERRA targets can be reused; training the image adapters still requires crops and gradients through image encoding.

The next cell writes an example configuration and prints a terminal command; it does not train. The trainer validates and saves `best.pt`. Its step budget and cell caps are illustrative, not tuned for this two-section cohort.

In [ ]:
if PREPARE_PAIRED:
    import shlex
    from cellbridge.config import load_config

    train_cfg = load_config(ROOT / "configs/train/lora/cellbridge.yaml")
    train_cfg.update({
        "run_name": "tutorial_hest",
        "data_root": str(PAIRED), "splits_path": str(PAIRED / "splits.csv"),
        "output_dir": str(WORK / "training"),
        "checkpoint_dir": str(WORK / "checkpoints/tutorial_hest"),
        "max_cells_per_sample": 5000, "max_val_cells_per_sample": 2000,
    })
    train_cfg["training"].update({"max_steps": 20000, "eval_every": 1000, "patience": 5})
    TRAIN_CONFIG = WORK / "train.yaml"
    TRAIN_CONFIG.write_text(yaml.safe_dump(train_cfg, sort_keys=False))
    print(shlex.join(["cellbridge-train", "--config", str(TRAIN_CONFIG)]))

For orientation, the core update uses these CellBridge APIs (illustrative code, **not executed**):

```python
from cellbridge.train.lora.data import build_datasets
from cellbridge.train.lora.model import build_uni2_distiller, contrastive_loss
from cellbridge.train.lora.train import make_loader, make_optimizer

student = build_uni2_distiller(**train_cfg['model']).to(DEVICE)
train_data, val_data = build_datasets(train_cfg, student.eval_transforms)
loader = make_loader(train_data, train_cfg, training=True)
optimizer, _ = make_optimizer(student, train_cfg)
student.train()
for batch in loader:
    optimizer.zero_grad(set_to_none=True)
    image_z = student(batch['image'].to(DEVICE))['projected']
    target_z = student.project_terra(batch['terra'].to(DEVICE))
    loss = contrastive_loss(image_z, target_z, temperature=0.07)['loss_total']
    loss.backward()
    optimizer.step()
train_data.close()
val_data.close()
```

For a real run use the printed command, which also validates and checkpoints. This command trains a new adapter from UNI2-h; it does not resume the released CellBridge adapter. To continue adapting released weights in a custom loop, initialize with `load_checkpoint(...)` instead and keep its architecture compatible with the teacher.

For the rest of this tutorial, we skip encoder training and load the released adapter. If you trained your own, set `LOCAL_CHECKPOINT` to its `best.pt` below. Re-extract all reference and query features when changing checkpoints.

## 3. Get UNI2-h and pretrained CellBridge weights

The adapter is a small checkpoint; it does **not** contain the UNI2-h backbone. Download both, then let CellBridge reconstruct the model using the architecture saved in the adapter. The released revision below matches the repository's inference configuration.

In [ ]:
from cellbridge.train.lora.model import load_checkpoint

LOCAL_CHECKPOINT = None  # Or Path("/path/to/your/best.pt")
RELEASE = "1c334ee0783be5aaae93a5562522d73fc92ee6c1"
uni_weights = hf_hub_download("MahmoodLab/UNI2-h", filename="pytorch_model.bin")
checkpoint_path = Path(LOCAL_CHECKPOINT) if LOCAL_CHECKPOINT else Path(hf_hub_download(
    "ogutsevda/cellbridge", filename="cellbridge.pt", revision=RELEASE,
))
model, checkpoint = load_checkpoint(checkpoint_path, device=DEVICE, weights_path=uni_weights)
model.eval()
print(checkpoint["model_config"])

## 4. Extract per-cell H&E features on NuCLS

NuCLS supplies H&E fields of view, cell coordinates, labels, and a fixed train/validation/test split. There is no paired expression input in this section. The command below downloads the archive and prepares its existing annotations and splits; subsequent runs reuse the archive. To use a local archive, append `"--archive", "/path/to/NuCLS_dataset.zip"` to the argument list.

For this example, retain the repository's **NuCLS crop convention: 224 source pixels around each cell**. This differs from HEST's physical-size training crops above. Keep the crop convention identical when fitting and applying a task classifier; assess changes of resolution or tissue on held-out data.

In [ ]:
subprocess.run([
    sys.executable, "-m", "cellbridge.inference.prepare.nucls", str(NUCLS),
], check=True)
metadata = pd.read_csv(NUCLS / "nucls_info.csv", dtype={"row_id": str, "cell_uid": str})
metadata = metadata.loc[metadata.label != "unlabeled"].copy()
assert metadata.row_id.is_unique and metadata.cell_uid.is_unique
assert metadata.groupby("slide_id").split.nunique().eq(1).all()
display(metadata.groupby("split").size().rename("cells").to_frame())

This helper only reads pixels and cell coordinates. It returns **1536-dimensional adapted image features** for classification. The model's **384-dimensional projection** was used for alignment during encoder training; it is not a cell-type probability vector.

In [ ]:
from cellbridge.inference.embed.crops import center_cell_crop

@torch.inference_mode()
def embed_cells(image, cells, batch_size=16):
    """Embed cells in input row order; no labels or expression are read."""
    features = []
    for start in range(0, len(cells), batch_size):
        rows = cells.iloc[start:start + batch_size]
        patches = [model.eval_transforms(center_cell_crop(image, row, size=224))
                   for _, row in rows.iterrows()]
        images = torch.stack(patches).to(DEVICE)
        features.append(model.encode_images(images, use_lora=True).float().cpu().numpy())
    return np.concatenate(features)

# Inspect one cell and the available representations without loading a second UNI model.
example = metadata.iloc[0]
with Image.open(NUCLS / "raw/rgb" / example.image_name) as source:
    image = np.asarray(source.convert("RGB"))
crop = center_cell_crop(image, example, size=224)
display(crop)
with torch.inference_mode():
    batch = model.eval_transforms(crop).unsqueeze(0).to(DEVICE)
    original_uni = model.encode_images(batch, use_lora=False)
    adapted = model.encode_images(batch, use_lora=True)
    projected = model.apply_heads(adapted)["projected"]
print("UNI:", tuple(original_uni.shape), "CellBridge:", tuple(adapted.shape),
      "Alignment projection:", tuple(projected.shape))

In [ ]:
# Encode one field of view at a time; lower batch_size above if GPU memory is limited.
feature_blocks, row_ids = [], []
for image_name, cells_in_image in metadata.groupby("image_name", sort=False):
    with Image.open(NUCLS / "raw/rgb" / image_name) as source:
        image = np.asarray(source.convert("RGB"))
    feature_blocks.append(embed_cells(image, cells_in_image[["x", "y"]]))
    row_ids.extend(cells_in_image.row_id)
X = np.concatenate(feature_blocks)
row_ids = np.asarray(row_ids, dtype=str)
# Explicit ID lookup preserves alignment even if metadata CSV rows are reordered.
aligned = metadata.set_index("row_id").loc[row_ids].reset_index()
assert X.shape == (len(aligned), 1536)
np.savez(WORK / "nucls_cellbridge.npz", embeddings=X, row_ids=row_ids,
         cell_ids=aligned.cell_uid.to_numpy(dtype=str))
del feature_blocks
print(f"Saved {len(X):,} cell features to {WORK / 'nucls_cellbridge.npz'}")

## 5. Fit a classifier for the task

For clarity, fit one standardized logistic-regression classifier with fixed `C=1`. Both the scaler and classifier see **training cells only**. Validation and test cells are used only for scoring; the test split must stay untouched by model or hyperparameter selection. This short example is not the paper's full probe-selection protocol.

With your own paired dataset, join your reference annotations to extracted image features by `(sample_id, cell_id)` and replace `y` below. Molecular measurements may supply reference labels, but they are not classifier inputs. Reserve whole donors/slides for validation and testing rather than randomly splitting neighboring cells.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = aligned.split.eq("train").to_numpy()
y = aligned.label.to_numpy(dtype=str)
classifier = make_pipeline(
    StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=0),
)
classifier.fit(X[train], y[train])
scores = []
for split in ("val", "test"):
    selected = aligned.split.eq(split).to_numpy()
    assert set(y[selected]) <= set(classifier.classes_)
    predicted = classifier.predict(X[selected])
    scores.append({
        "split": split,
        "macro_f1": f1_score(y[selected], predicted, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y[selected], predicted),
    })
display(pd.DataFrame(scores).round(3))

## 6. Predict from H&E and cell locations alone

Use one held-out field of view as a stand-in for a new image. Drop its labels before encoding; attach them only afterward if you want to evaluate the predictions. In your own application, supply an RGB image and a table with **`cell_id, x, y`**, where `x, y` are cell centers in that image's pixel coordinates. A cell detector supplies these locations on a new slide.

The fitted classifier can only assign labels in its training vocabulary. Its maximum probability is a score, not a calibrated reliability guarantee. For ontology-aware prediction sets, continue with section 7.

In [ ]:
query_name = aligned.loc[aligned.split == "test", "image_name"].iloc[0]
query_cells = aligned.loc[aligned.image_name == query_name, ["cell_uid", "x", "y"]].rename(
    columns={"cell_uid": "cell_id"},
)
with Image.open(NUCLS / "raw/rgb" / query_name) as source:
    query_image = np.asarray(source.convert("RGB"))

# Neither query_cells nor embed_cells contains labels, counts, or TERRA features.
query_features = embed_cells(query_image, query_cells)
probabilities = classifier.predict_proba(query_features)
predictions = query_cells.copy()
predictions["predicted_label"] = classifier.classes_[probabilities.argmax(axis=1)]
predictions["max_probability"] = probabilities.max(axis=1)
predictions.to_csv(WORK / "he_only_predictions.csv", index=False)
np.savez(WORK / "he_only_probabilities.npz",
         cell_ids=predictions.cell_id.to_numpy(dtype=str),
         classes=classifier.classes_.astype(str), probabilities=probabilities)
display(predictions.head())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(query_image)
colours = plt.get_cmap("tab20")
for i, (label, cells_in_class) in enumerate(predictions.groupby("predicted_label")):
    ax.scatter(cells_in_class.x, cells_in_class.y, label=label,
               color=colours(i), s=18, edgecolors="black", linewidths=0.2)
ax.set_title("CellBridge predictions from H&E")
ax.axis("off")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Predicted cell type")
plt.show()

Save the task classifier together with its preprocessing and encoder settings. Reuse **the same adapter, UNI2-h backbone, feature representation, and crop convention** for new cells. The classifier file does not contain either encoder's weights.

In [ ]:
import joblib

joblib.dump({
    "classifier": classifier,
    "adapter_path": str(checkpoint_path),
    "adapter_repo": None if LOCAL_CHECKPOINT else "ogutsevda/cellbridge",
    "adapter_revision": None if LOCAL_CHECKPOINT else RELEASE,
    "uni_weights_path": uni_weights,
    "representation": "lora_adapted",
    "crop": {"source_size": 224, "fill": 255},
}, WORK / "nucls_classifier.joblib")
print(f"Classifier, features, and predictions saved under {WORK}")

## 7. Optional: calibrate Cell-Ontology-Fallback (COF)

With `RUN_COF = True`, reuse the image features from section 4 to fit and calibrate ontology-aware prediction sets. No additional image encoding or TERRA inference is needed.

COF uses a **separate classifier** because it maps the task labels into the ontology and reserves validation cells for calibration. Its PCA and classifier are fitted on training cells; classifier regularization is selected within training data. Ontology mapping and minimum class support can exclude some cells. These predictions therefore need not match the simple classifier in section 5.

We use the supplied NuCLS ontology mapping and a **90% target coverage** (`alpha=0.1`). The reported test coverage is empirical; the target is not a guarantee for an arbitrary new tissue or shifted dataset. `branch_graph` is CellBridge's COF method.

In [ ]:
if RUN_COF:
    from cellbridge.config import load_config
    from cellbridge.inference.conformal import conformal_ood
    from cellbridge.ontology import load_mapping, load_ontology

    cfg = load_config(ROOT / "configs/inference/nucls.yaml")
    settings = cfg["conformal"]
    mapping = load_mapping(settings["mapping"])
    ontology = load_ontology(settings["ontology"])
    task = conformal_ood.prepare_task(
        X, aligned, mapping, task="all", arm="lora_adapted",
        pca_components=cfg["probe"]["pca_components"],
        min_class_count_per_split=settings["min_class_count_per_split"],
        seed=cfg["probe"]["seed"],
    )
    cof_result = conformal_ood.evaluate(
        task, "linear", cfg["probe"]["linear"], mapping, ontology,
        branch_roots=settings["branch_roots"], alpha=settings["alpha"],
    )
    COF = WORK / "results/nucls/conformal_prediction"
    conformal_ood.save_result(COF, cof_result)
    cof_metrics = pd.read_csv(COF / "metrics.csv")
    display(cof_metrics.loc[cof_metrics.conformal_method == "branch_graph", [
        "marginal_coverage", "mean_prediction_set_size", "singleton_rate",
    ]].round(3))

### Inspect cell-level prediction sets

A singleton set retains one label; a larger set retains multiple plausible labels and can support a broader ontology category. Join predictions back to the image using saved cell identities.

The plot retains the original tutorial's set-size score, `1 − |C| / |Y|`, to show how specific each prediction is. It is not the probability that the prediction is correct. Empty sets are marked separately.

In [ ]:
if RUN_COF:
    with np.load(COF / "predictions.npz", allow_pickle=False) as saved:
        classes = saved["classes"].astype(str)
        sets = saved["branch_graph_sets"]
        cof_predictions = pd.DataFrame({
            "row_id": saved["ids"].astype(str),
            "true_label": saved["y_true"].astype(str),
            "prediction_set": [", ".join(classes[mask]) for mask in sets],
            "set_size": sets.sum(axis=1),
        })

    # Join by saved cell identity, not by CSV row order.
    cof_predictions = cof_predictions.merge(
        metadata[["row_id", "image_name"]], on="row_id", validate="one_to_one",
    )
    display(cof_predictions.head(10))
    print(f"COF results are saved under {COF}")

In [ ]:
if RUN_COF:
    # Qualitative inspection: paper Eq. 10, R = 1 - |C| / |Y|.
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    with np.load(COF / "predictions.npz", allow_pickle=False) as saved:
        top1 = pd.DataFrame({
            "row_id": saved["ids"].astype(str),
            "predicted_label": classes[saved["probabilities"].argmax(axis=1)],
        })
    view = cof_predictions.merge(top1, on="row_id", validate="one_to_one").merge(
        metadata[["row_id", "x", "y"]], on="row_id", validate="one_to_one",
    )
    view["reliability"] = 1 - view["set_size"] / len(classes)
    image_name = view["image_name"].value_counts().index[0]  # Change to inspect another image.
    cells = view.loc[view["image_name"] == image_name]
    with Image.open(NUCLS / "raw/rgb" / image_name) as image:
        tissue = np.asarray(image.convert("RGB"))

    fig, axes = plt.subplots(1, 4, figsize=(17, 5), layout="constrained")
    for ax, title in zip(axes, ["H&E", "Reference labels", "Predicted labels (top-1)", "Reliability score"]):
        ax.imshow(tissue)
        ax.set_title(title)
        ax.axis("off")
    colours = plt.get_cmap("tab20", len(classes))
    for ax, column in zip(axes[1:3], ["true_label", "predicted_label"]):
        codes = pd.Categorical(cells[column], categories=classes).codes
        ax.scatter(cells.x, cells.y, c=codes, cmap=colours, vmin=-0.5,
                   vmax=len(classes) - 0.5, s=24, edgecolors="black", linewidths=0.3)
    nonempty = cells["set_size"] > 0
    points = axes[3].scatter(cells.loc[nonempty, "x"], cells.loc[nonempty, "y"],
                             c=cells.loc[nonempty, "reliability"], cmap="RdYlGn",
                             vmin=0, vmax=1, s=24, edgecolors="black", linewidths=0.3)
    if (~nonempty).any():
        axes[3].scatter(cells.loc[~nonempty, "x"], cells.loc[~nonempty, "y"],
                        c="gray", marker="x", s=30, label="Empty set")
        axes[3].legend(loc="lower right")
    fig.colorbar(points, ax=axes[3], label="1 − set size / number of classes", shrink=0.7)
    handles = [Line2D([], [], marker="o", linestyle="", color=colours(i), label=label)
               for i, label in enumerate(classes)]
    fig.legend(handles=handles, loc="outside lower center", ncol=4, title="Cell types")
    fig.suptitle(f"{image_name} · {len(cells)} evaluated test cells")
    plt.show()

Outputs are saved under `outputs/tutorial/`:

- `nucls_cellbridge.npz`: image features and cell identities.
- `nucls_classifier.joblib`: the section 5 classifier and encoder/crop settings.
- `he_only_predictions.csv` and `he_only_probabilities.npz`: section 6 predictions.
- `results/nucls/conformal_prediction/`: optional COF metrics, prediction sets, and graph anchors.

To apply this to your own task:

1. Register paired expression and H&E data, preserve cell identity, and export the TERRA NPZ files from section 2.3.
2. Train an encoder with the section 2.4 command, or use released CellBridge weights. Encoder adaptation needs paired data; fitting a classifier on released image features needs only labeled images.
3. Extract reference image features and fit a classifier for your annotations using independent donor/slide splits.
4. Detect cells in new H&E images, crop them using the same convention, and apply that encoder and classifier. Neither TERRA nor expression data is needed here.

For whole-slide H&E, use `open_wsi` and `crop_cell` from `cellbridge.train.lora.data` to read regions lazily. For a HEST-style task, use the 112 µm field of view for **both** reference and query cells and refit the classifier. Do not apply the NuCLS classifier to a different crop convention without validation.

See the [TERRA tutorials](https://terra-st.readthedocs.io/en/latest/tutorials.html) for additional uses of the molecular embeddings, and the [reproducibility guide](../REPRODUCIBILITY.md) for full CellBridge experiments.